<a href="https://colab.research.google.com/github/ankorn/redred/blob/main/model/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# seq2seq

In [ ]:
!pip uninstall datasets torchao

Found existing installation: datasets 3.6.0
Uninstalling datasets-3.6.0:
  Would remove:
    /usr/local/bin/datasets-cli
    /usr/local/lib/python3.12/dist-packages/datasets-3.6.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/datasets/*
Proceed (Y/n)? y
  Successfully uninstalled datasets-3.6.0
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/torchao-0.10.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torchao/*
Proceed (Y/n)? y
  Successfully uninstalled torchao-0.10.0


In [ ]:
!pip -q install datasets==3.6.0 torchao==0.16.0 evaluate rouge_score

  Preparing metadata (setup.py) ... done


In [ ]:
from datasets import load_dataset

dataset = load_dataset("webis/tldr-17", split="train", streaming=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "sshleifer/distilbart-cnn-6-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/262 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # for BART; adjust for other models
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 334,053,376 || trainable%: 0.3531


In [ ]:
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["content"],
        max_length=1024, # CHECK: BART's max position embedding is 1024
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        examples["summary"],
        max_length=128, # TODO: CHECK
        truncation=True,
        padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)

train_dataset = tokenized_dataset.shuffle().take(100000)
eval_dataset = tokenized_dataset.shuffle().take(1000)

In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)

    # BERTScore (optional, slower)
    # from bert_score import score
    # P, R, F1 = score(decoded_preds, decoded_labels, lang="en")
    # result["bertscore"] = F1.mean().item()

    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
training_args = Seq2SeqTrainingArguments(
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=100,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=128,
    metric_for_best_model="eval_rouge1",
    load_best_model_at_end=True,
    max_steps=1000
)

trainer = Seq2SeqTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
100,8.934473,1.643555,0.135700,0.025000,0.097600,0.099300
200,8.073896,1.597656,0.134200,0.024500,0.096000,0.097700
300,7.962656,1.578125,0.134700,0.024200,0.096700,0.098500
400,7.610508,1.568359,0.135200,0.024700,0.096500,0.098600


KeyboardInterrupt: 

In [ ]:
post = '''Why do all old people have short hair?
I’ve noticed something and I’m curious if there’s a reason behind it. Most older women I see (like 70+) tend to have short hair. I honestly can’t think of many I’ve met with long hair.
Is there a practical reason for this? Like easier maintenance, hair thinning, or just preference over time? Would love to hear from anyone who knows or has experience with this.'''

In [ ]:
model.eval()
inputs = tokenizer(post, return_tensors="pt", max_length=512, truncation=True)
summary_ids = model.generate(inputs["input_ids"], max_length=128, num_beams=4, early_stopping=True)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)

 Most older women (like 70+) tend to have short hair . Is there a practical reason for this? Like easier maintenance, hair thinning, or just preference over time? Would love to hear from anyone who knows or has experience with this. Would like to see anyone


# chat llm

In [ ]:
%pip -q install -U transformers trl

%pip uninstall -q -y torchao

%pip -q install -q trl evaluate datasets==3.6.0 trl==0.12.0 bitsandbytes>=0.46.1 rouge_score peft torchao==0.16.0 langdetect

import argparse
import json
import os
import random
import re
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer
import evaluate
from tqdm import tqdm

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(os.environ["PYTORCH_CUDA_ALLOC_CONF"])

!nvidia-smi

expandable_segments:True
Sun May 17 18:48:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   37C    P0             53W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------------

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # ~1.1 GB Q4_K_M
# MODEL_NAME = "google/gemma-3-1b-it"       # ~600 MB Q4_K_M

NUM_SAMPLES = 150_000

MAX_SEQ_LENGTH = 1024

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1

BATCH_SIZE = 8
GRAD_ACCUMULATION = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
WARMUP_RATIO = 0.03

In [ ]:
from pandas import DataFrame
from datasets import Dataset

dataset = load_dataset("webis/tldr-17", split="train", trust_remote_code=True, streaming=True)

df = DataFrame(list(dataset.take(NUM_SAMPLES)))

df = df.drop_duplicates(subset=['content', 'summary'])
print('removed', NUM_SAMPLES - len(df), 'duplicates')

init_len = len(df)
df = df[df['content'].str.len() < 2500]
print('removed', init_len - len(df), 'too long content')

dataset = Dataset.from_pandas(df)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


removed 249 duplicates
removed 12482 too long content


In [ ]:
from langdetect import detect

init_len = len(dataset)

def filter_batch(batch):
    return [detect(content) == 'en' for content in batch["content"]]

dataset = dataset.filter(filter_batch, batched=True, batch_size=1000)

print('removed', init_len - len(dataset), 'non-english content')

Filter:   0%|          | 0/137269 [00:00<?, ? examples/s]

removed 705 non-english content


In [ ]:
dataset = dataset.train_test_split(test_size=0.005)
train_ds = dataset["train"]
eval_ds = dataset["test"]

def clean_text(text: str) -> str:
    text = text.replace("&#x200B;", "").replace("&gt;", ">").replace("&lt;", "<")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

system_msg = (
    "You are a helpful assistant that writes concise TL;DR summaries "
    "of Reddit posts in one or two sentences."
)

def format_chat(example):
    content = clean_text(example["content"])
    summary = clean_text(example["summary"])

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": f"Reddit post:\n{content}"},
        {"role": "assistant", "content": summary},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

train_ds_formatted = train_ds.map(format_chat, remove_columns=train_ds.column_names)
eval_ds_formatted  = eval_ds.map(format_chat, remove_columns=eval_ds.column_names)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Map:   0%|          | 0/136582 [00:00<?, ? examples/s]

Map:   0%|          | 0/687 [00:00<?, ? examples/s]

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


In [ ]:
from transformers import GenerationConfig

eval_generation_config = GenerationConfig(
    max_new_tokens=64,
    do_sample=False,
    num_beams=1,
    early_stopping=True,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")],
)

full_generation_config = GenerationConfig(
    max_new_tokens=64,
    do_sample=False,
    num_beams=4,
    early_stopping=True,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")],
)

class EvalTrainer(SFTTrainer):
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        if prediction_loss_only:
            return super().prediction_step(model, inputs, True, ignore_keys)

        # 1. Compute loss (teacher-forced)
        loss, _, _ = super().prediction_step(model, inputs, True, ignore_keys)

        # 2. Ensure eval mode + clear cache before generation
        model.eval()
        torch.cuda.empty_cache()

        input_ids = inputs["input_ids"]
        attention_mask = inputs.get("attention_mask")
        labels = inputs.get("labels")
        batch_size = input_ids.shape[0]
        device = input_ids.device

        # 3. Extract prompts (same logic as before)
        prompt_lengths = []
        prompt_ids_list = []
        prompt_mask_list = []

        for i in range(batch_size):
            if attention_mask is not None:
                active = attention_mask[i].nonzero(as_tuple=True)[0]
                actual_start = active[0].item() if len(active) > 0 else 0
            else:
                actual_start = 0

            if labels is not None:
                valid = (labels[i] != -100).nonzero(as_tuple=True)[0]
                completion_start = valid[0].item() if len(valid) > 0 else input_ids.shape[1]
            else:
                completion_start = input_ids.shape[1]

            pl = completion_start - actual_start
            prompt_lengths.append(pl)
            prompt_ids_list.append(input_ids[i, actual_start:completion_start])
            if attention_mask is not None:
                prompt_mask_list.append(attention_mask[i, actual_start:completion_start])

        # 4. Left-pad prompts to batch max (generation requires uniform length)
        max_pl = max(prompt_lengths) if prompt_lengths else 0
        prompt_ids = torch.full((batch_size, max_pl), self.processing_class.pad_token_id,
                                dtype=torch.long, device=device)
        prompt_mask = torch.zeros((batch_size, max_pl), dtype=torch.long, device=device)

        for i in range(batch_size):
            pl = prompt_lengths[i]
            prompt_ids[i, -pl:] = prompt_ids_list[i]  # right-align / left-pad
            if attention_mask is not None:
                prompt_mask[i, -pl:] = prompt_mask_list[i]

        # 5. Generate with eval config (greedy, beam=1)
        with torch.no_grad():
            outputs = model.generate(
                input_ids=prompt_ids,
                attention_mask=prompt_mask,
                generation_config=eval_generation_config,
            )

        # 6. Strip prompts, keep only new tokens
        gen_tokens = []
        max_gen = 0
        for i in range(batch_size):
            gen_only = outputs[i, max_pl:]  # remove the padded prompt length
            gen_tokens.append(gen_only)
            max_gen = max(max_gen, gen_only.shape[0])

        padded_preds = torch.full((batch_size, max_gen), self.processing_class.pad_token_id,
                                  dtype=torch.long, device=device)
        for i in range(batch_size):
            padded_preds[i, :gen_tokens[i].shape[0]] = gen_tokens[i]

        # 7. Extract completion-only labels
        if labels is not None:
            label_tokens = []
            max_lab = 0
            for i in range(batch_size):
                valid = labels[i][labels[i] != -100]
                label_tokens.append(valid)
                max_lab = max(max_lab, valid.shape[0])

            padded_labels = torch.full((batch_size, max_lab), -100,
                                       dtype=torch.long, device=device)
            for i in range(batch_size):
                padded_labels[i, :label_tokens[i].shape[0]] = label_tokens[i]
        else:
            padded_labels = None

        return (loss, padded_preds, padded_labels)

[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
# TODO: try per_device_eval_batch_size=1 without EvalTrainer

In [ ]:
from transformers import EarlyStoppingCallback
import numpy as np
import evaluate
from trl import DataCollatorForCompletionOnlyLM

model.generation_config = eval_generation_config

metric = evaluate.load("rouge")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    predictions = predictions.astype(np.int32)
    labels = labels.astype(np.int32)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    print('>>>', decoded_preds[0], '\n')
    print(decoded_labels[0], '\n')

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {k: round(v, 4) for k, v in result.items()}

training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=GRAD_ACCUMULATION,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),

    logging_steps=1,
    eval_strategy="steps",
    eval_steps=1,
    save_strategy="steps",
    save_steps=1,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_rouge1",
    greater_is_better=True,

    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    report_to="none",
)

# trainer = EvalTrainer(
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds_formatted,
    eval_dataset=eval_ds_formatted.take(50),

    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    data_collator=DataCollatorForCompletionOnlyLM(
        response_template="<|im_start|>assistant\n",
        tokenizer=tokenizer
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    compute_metrics=compute_metrics,
)

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/136582 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
rouge = evaluate.load("rouge")
model.eval()

predictions, references = [], []

eval_subset = eval_ds.shuffle(seed=42).select(range(min(15, len(eval_ds))))

for example in tqdm(eval_subset, desc="Generating summaries"):
    content = clean_text(example["content"])
    ref = clean_text(example["summary"])

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": f"Reddit post:\n{content}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **full_generation_config,
        )

    pred = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
    ).strip()

    predictions.append(pred)
    references.append(ref)

results = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True,
)

for k, v in results.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
import huggingface_hub

huggingface_hub.login()

model.push_to_hub("pameydorke/redred-qwen2.5-1.5-lora")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  51%|#####1    |  152MB /  295MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/pameydorke/redred-qwen2.5-1.5-lora/commit/8710273e5d60c92ead355ac185932e850c3defed', commit_message='Upload model', commit_description='', oid='8710273e5d60c92ead355ac185932e850c3defed', pr_url=None, repo_url=RepoUrl('https://huggingface.co/pameydorke/redred-qwen2.5-1.5-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='pameydorke/redred-qwen2.5-1.5-lora'), pr_revision=None, pr_num=None)

In [ ]:
print(predictions[0], '\n', references[0], '\n')
print(predictions[13], '\n', references[13], '\n')
print(predictions[11], '\n', references[11], '\n')

Practice, communication, practice, communication. 
 Keep fucking, it gets better. =P 

Guns, bullets, missiles, chemical weapons, biological weapons, tanks. 
 those that profit by death 

I don't know what to do with my penis. 
 spacedicks lost its "charm" because caps-lock became silly and repeated gore lost its value. 



In [ ]:
print(predictions[1], '\n', references[1], '\n')
print(predictions[14], '\n', references[14], '\n')
print(predictions[12], '\n', references[12], '\n')

Emotions are not of less integrity or value simply because one chooses them rather then being slave to them. In other words, if you have achieved sufficient rational control over the affect that your emotions have on you that you no longer are swept up by them, then that only means that you may choose when to indulge in them 
 Repression is not control, repression is just another kind of failure to control. 

I don't pirate movies because I don't watch them. I don't pirate songs because I don't listen to them. I don't pirate software because I don't use it. 
 IMHO the Christian doesn't pirate not because piracy is wrong, but because he has better things to do with his time. 

I'm not trying to change your mind. I'm trying to help you understand his point of view. 
 The new perspective you're looking for is your husband's. Until you honestly look into his opinion, this decision should not be made. This doesn't mean you'll leave though. But it may help you understand where he's coming fr

In [ ]:
# 1. Merge LoRA into base model (do this before ONNX export)
merged_model = model.merge_and_unload()

# 2. Save merged model temporarily
import tempfile, os
temp_dir = tempfile.mkdtemp()
merged_model.save_pretrained(temp_dir)
tokenizer.save_pretrained(temp_dir)

# 3. Export to ONNX using Optimum
!pip install -q optimum[onnxruntime]

from optimum.onnxruntime import ORTModelForCausalLM
from transformers import AutoTokenizer

onnx_temp = tempfile.mkdtemp()

# Export with cache for efficient inference
ort_model = ORTModelForCausalLM.from_pretrained(
    temp_dir,
    export=True,
    provider="CUDAExecutionProvider",  # or "CPUExecutionProvider"
    use_cache=True,
)

ort_model.save_pretrained(onnx_temp)
tokenizer.save_pretrained(onnx_temp)

# 4. Push to Hugging Face Hub
from huggingface_hub import HfApi, create_repo

repo_id = "pameydorke/redred-qwen2.5-1.5-lora-onnx"
create_repo(repo_id, exist_ok=True, repo_type="model")

api = HfApi()
api.upload_folder(
    folder_path=onnx_temp,
    repo_id=repo_id,
    repo_type="model",
)

print(f"Pushed to https://huggingface.co/{repo_id}")

In [ ]:
model.eval()

post_text = '''Why does Black Pepper get the place of honor next to Salt?
Salt is a substance that is literally necessary to keep us alive. Because of that, it makes sense that it is one of the five tastes that our tounge is able to detect. it basically just makes meat taste better on like an evolutionary level.

And then the thing that is always said in the same breath is just a seemingly random spice made from spicy ground up seeds or something. I assume that not all cultures value black pepper the same as we do in the west. But since it doesnt have the organic significance that salt does, I assume it could be any spicy thing we pair with salt. Why is it specifically black pepper?
'''

messages = [
    {"role": "system", "content": "Write a concise TL;DR summary of the Reddit post."},
    {"role": "user", "content": f"Reddit post:\n{post_text}"},
]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )

summary = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
).strip()

summary

"Black pepper is not necessarily more flavorful than other spices because its flavor profile isn't particularly unique. The flavor profile of black pepper would probably be similar to cinnamon and cloves if you were to try them out yourself. They are both sweet, warm, and slightly spicier. So maybe they're not so different after all"

In [ ]:
adapted_path = 'https://huggingface.co/pameydorke/redred-summarizer'

In [ ]:
# (?)

peft_model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = model.merge_and_unload()

from transformers import AutoModelForCausalLM
merged = AutoModelForCausalLM.from_pretrained("./tldr-lora-final")
merged.save_pretrained("./tldr-merged")

# 3. Convert to GGUF (install llama.cpp, then)
python convert_hf_to_gguf.py ./tldr-merged --outfile tldr-qwen-1.5b-q4_k_m.gguf --outtype q4_k_m

In [ ]:
# Merge your LoRA adapter into the base Qwen2.5-1.5B-Instruct.
# Export the merged model to ONNX with optimum-cli :
!optimum-cli export onnx --model ./tldr-merged ./tldr-onnx/
# Quantize for the browser:
!optimum-cli onnxruntime quantize --onnx_model ./tldr-onnx/ -o ./tldr-onnx-q4/

# JavaScript
# import { pipeline } from "@huggingface/transformers";
# const generator = await pipeline(
#   "text-generation",
#   "your-username/tldr-qwen-onnx",
#   { dtype: "q4", device: "webgpu" }
# );